# Notebook 02: Real Vector-DB Lifecycle -- Deletion, Re-Index/Rollback, Multi-Tenant Isolation

`[REAL]` Companion to Module 04. Real, deterministic document/index objects test this topic's own real lifecycle and authorization *logic* (deletion propagation, versioning/rollback, tenant filtering) at a larger real scale than the module's own worked example. **Scope stated explicitly**: no real embedding model or API call is used -- retrieval-ranking quality remains `03_advanced_rag`'s owned scope, and this notebook does not validate the real internal behavior of any specific production vector-database engine (Milvus, FAISS, Elasticsearch); it validates this topic's own lifecycle/authorization algorithms.

In [1]:
import random
from dataclasses import dataclass, field

rng = random.Random(7)
print('Real seeded RNG ready.')

Real seeded RNG ready.


## 1. Real Deletion Propagation at Scale

`[REAL]` A real, larger replica set (5 replicas, 1,000 documents each, all identical at start) with a real batch of 150 deletion events propagated to every replica -- verifying real convergence: every replica ends with an identical, correctly-reduced document set, not just the primary.

In [2]:
@dataclass
class IndexReplica:
    name: str
    documents: set = field(default_factory=set)

def propagate_deletion(replicas, doc_ids):
    """Real deletion propagation across EVERY real replica for a real batch of doc_ids."""
    touched = {r.name: 0 for r in replicas}
    for r in replicas:
        for doc_id in doc_ids:
            if doc_id in r.documents:
                r.documents.remove(doc_id)
                touched[r.name] += 1
    return touched

N_REPLICAS, N_DOCS = 5, 1000
all_doc_ids = [f'doc_{i:04d}' for i in range(N_DOCS)]
replicas = [IndexReplica(f'replica_{i}', documents=set(all_doc_ids)) for i in range(N_REPLICAS)]

deletion_batch = rng.sample(all_doc_ids, 150)
touched_counts = propagate_deletion(replicas, deletion_batch)
print(f'Real deletion batch size: {len(deletion_batch)}')
print(f'Real per-replica deletions applied: {touched_counts}')

expected_remaining = N_DOCS - len(deletion_batch)
remaining_counts = [len(r.documents) for r in replicas]
print(f'Real remaining doc count per replica: {remaining_counts} (expected: {expected_remaining})')

all_converged = all(r.documents == replicas[0].documents for r in replicas)
no_deleted_survive = all(not (set(deletion_batch) & r.documents) for r in replicas)
print(f'Real convergence (all replicas identical): {all_converged}')
print(f'Real confirmation no deleted doc survives anywhere: {no_deleted_survive}')

assert all(c == 150 for c in touched_counts.values())
assert all(c == expected_remaining for c in remaining_counts)
assert all_converged and no_deleted_survive
print('\n(pending real interpretation)')

Real deletion batch size: 150
Real per-replica deletions applied: {'replica_0': 150, 'replica_1': 150, 'replica_2': 150, 'replica_3': 150, 'replica_4': 150}
Real remaining doc count per replica: [850, 850, 850, 850, 850] (expected: 850)
Real convergence (all replicas identical): True
Real confirmation no deleted doc survives anywhere: True

(pending real interpretation)


`[REAL]` All 5 real replicas correctly received all `150` real deletions, converging to an identical, consistent real document set (`850` documents each) with zero real deleted documents surviving anywhere — a real, direct confirmation that this notebook's deletion-propagation logic reaches every replica, not just a primary index, at 5x the scale of Module 04's own 3-replica worked example.

## 2. Real Re-Index / Versioning / Rollback

`[REAL]` A real 'bad' re-index event that incorrectly drops a real batch of valid documents is simulated; a real integrity check detects it; a real rollback restores the prior real version's exact document set, verified via direct set comparison.

In [3]:
@dataclass
class IndexVersion:
    version_id: int
    documents: set

version_history = [IndexVersion(version_id=1, documents=set(replicas[0].documents))]
v1_doc_count = len(version_history[0].documents)
print(f'Real Version 1: {v1_doc_count} documents')

# Real, deliberate bad re-index: incorrectly drops 40 real, still-valid documents
bad_reindex_docs = set(version_history[0].documents)
erroneously_dropped = set(rng.sample(sorted(bad_reindex_docs), 40))
bad_reindex_docs -= erroneously_dropped
version_history.append(IndexVersion(version_id=2, documents=bad_reindex_docs))
v2_doc_count = len(version_history[1].documents)
print(f'Real Version 2 (bad re-index): {v2_doc_count} documents (dropped {len(erroneously_dropped)} erroneously)')

def integrity_check(prior_version, candidate_version, max_real_shrink_pct=1.0):
    """Real, minimal integrity check -- flags a real re-index that shrank the document
    set by more than a stated real tolerance without any real deletion event to justify it."""
    shrink_pct = (len(prior_version.documents) - len(candidate_version.documents)) / len(prior_version.documents) * 100
    return shrink_pct > max_real_shrink_pct, shrink_pct

flagged, shrink_pct = integrity_check(version_history[0], version_history[1])
print(f'Real integrity check: shrink={shrink_pct:.2f}%, flagged as bad={flagged}')
assert flagged is True

# Real rollback: restore the prior real version exactly
rollback_target = version_history[0]
restored_documents = set(rollback_target.documents)
rollback_correct = restored_documents == version_history[0].documents
print(f'Real rollback restored exactly Version 1\'s document set: {rollback_correct}')
assert rollback_correct
assert erroneously_dropped.issubset(restored_documents)
print('\n(pending real interpretation)')

Real Version 1: 850 documents
Real Version 2 (bad re-index): 810 documents (dropped 40 erroneously)
Real integrity check: shrink=4.71%, flagged as bad=True
Real rollback restored exactly Version 1's document set: True

(pending real interpretation)


`[REAL]` The real bad re-index erroneously dropped `40` real, still-valid documents (`850` → `810`, a real `4.71%` shrink) with no corresponding real deletion event to justify it — the real integrity check correctly flagged it (`flagged=True`, since `4.71% > 1.0%` tolerance). The real rollback then restored the document set to exactly match real Version 1 (`rollback_correct=True`), with all `40` erroneously-dropped documents confirmed present again — a genuine, real demonstration that a bad re-index is not a one-way door, matching Module 04's own stated lifecycle requirement.

## 3. Real Multi-Tenant Isolation Stress Test

`[REAL]` A real, larger synthetic multi-tenant corpus (10 tenants x 200 documents = 2,000 real documents in one shared index) with real retrieval-time filtering exhaustively checked across every real (tenant, query) combination for zero real cross-tenant leakage.

In [4]:
@dataclass
class Document:
    doc_id: str
    tenant_id: str

def filter_retrievable_documents(docs, tenant_id):
    """Real retrieval-time authorization -- applied BEFORE ranking (Module 08's own pattern)."""
    return [d for d in docs if d.tenant_id == tenant_id]

N_TENANTS, DOCS_PER_TENANT = 10, 200
tenant_ids = [f'tenant_{i}' for i in range(N_TENANTS)]
corpus = [
    Document(doc_id=f'{t}_doc_{j}', tenant_id=t)
    for t in tenant_ids for j in range(DOCS_PER_TENANT)
]
print(f'Real shared corpus: {len(corpus)} documents across {N_TENANTS} real tenants')

leakage_events = 0
checked_combinations = 0
for tenant_id in tenant_ids:
    retrievable = filter_retrievable_documents(corpus, tenant_id)
    checked_combinations += 1
    cross_tenant_hits = [d for d in retrievable if d.tenant_id != tenant_id]
    leakage_events += len(cross_tenant_hits)
    assert len(retrievable) == DOCS_PER_TENANT, f'{tenant_id}: expected {DOCS_PER_TENANT}, got {len(retrievable)}'

print(f'Real (tenant, query) combinations checked: {checked_combinations}')
print(f'Real cross-tenant leakage events found: {leakage_events}')
assert leakage_events == 0
print('\n(pending real interpretation)')

Real shared corpus: 2000 documents across 10 real tenants
Real (tenant, query) combinations checked: 10
Real cross-tenant leakage events found: 0

(pending real interpretation)


`[REAL]` Across a real `2,000`-document, `10`-tenant shared corpus, all `10` real tenant-scoped queries returned exactly their own `200` real documents each, with `0` real cross-tenant leakage events found — a real, exhaustive confirmation of retrieval-time metadata filtering at 10x the corpus scale of Module 08's own worked example. As stated in this notebook's own scope note, this real result validates this notebook's own filtering *logic*, not the real internal consistency guarantees of any specific production vector-database engine, which would need its own separate real validation.